**Notes**

TCGA pipeline => filter by cancer type between EGFR altered vs unaltered

EGFR_mut = patients with EGFR mutation
EGFR_wt = patients without mutation

lambda instead of for looops

- need expression, mutation, and clinical data

[CBIOPORTAL (Lung Adenocarcinoma with EGFR)](https://www.cbioportal.org/study/summary?id=luad_tcga_pan_can_atlas_2018&plots_horz_selection=%7B%22selectedGeneOption%22%3A3024%2C%22dataType%22%3A%22COPY_NUMBER_ALTERATION%22%2C%22selectedDataSourceOption%22%3A%22log2CNA%22%7D&plots_vert_selection=%7B%22selectedGeneOption%22%3A1956%2C%22dataType%22%3A%22MUTATION_EXTENDED%22%2C%22mutationCountBy%22%3A%22MutationType%22%7D&plots_coloring_selection=%7B%7D)

In [68]:
#pip install pandas scikit-learn
import pandas as pd

In [69]:
cancer = pd.read_csv('checkpoint_3.csv')
#set missing values to NaN
cancer.replace('?', pd.NA, inplace=True)
cancer['gender_encoded'] = cancer['Sex'].map({'Female': 0, 'Male': 1})
cancer_EGFR = cancer.drop(columns=['Male','Female','Study ID', 'American Joint Committee on Cancer Publication Version Type', 'American Joint Committee on Cancer Metastasis Stage Code'])                               

cancer_EGFR.head()

,Sample ID,Patient ID,Diagnosis Age,Neoplasm Disease Stage American Joint Committee on Cancer Code,Aneuploidy Score,Buffa Hypoxia Score,Cancer Type,TCGA PanCanAtlas Cancer Type Acronym,Cancer Type Detailed,Last Communication Contact from Initial Pathologic Diagnosis Date,...,Tissue Prospective Collection Indicator,Tissue Retrospective Collection Indicator,Tissue Source Site,Tissue Source Site Code,TMB (nonsynonymous),Tumor Disease Anatomic Site,Tumor Type,Patient Weight,Winter Hypoxia Score,gender_encoded
0,TCGA-05-4402-01,TCGA-05-4402,57,STAGE IV,31,17,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,No,Yes,Indivumed,5,4.033333,Lung,"Lung Adenocarcinoma, Mixed Subtype",NaN,24,0
1,TCGA-38-6178-01,TCGA-38-6178,70,STAGE IIIA,22,-5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,448.0,...,Yes,No,UNC,38,2.233333,Lung,Lung Adenocarcinoma (NOS),NaN,4,0
2,TCGA-44-A4SU-01,TCGA-44-A4SU,67,STAGE IA,12,-11,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,76.0,...,Yes,No,Christiana Healthcare,44,4.433333,Lung,Lung Adenocarcinoma (NOS),NaN,18,0
3,TCGA-49-4501-01,TCGA-49-4501,67,STAGE IB,19,5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,1421.0,...,No,Yes,Johns Hopkins,49,1.600000,Lung,Lung Adenocarcinoma (NOS),NaN,12,0
4,TCGA-50-6591-01,TCGA-50-6591,63,STAGE IV,28,21,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,No,Yes,University of Pittsburgh,50,1.866667,Lung,Lung Adenocarcinoma (NOS),NaN,32,0


In [70]:
#Disease-specific Survival Analysis separate 0 as alive and 1 as dead
cancer_EGFR['Disease-specific Survival status'] = cancer_EGFR['Disease-specific Survival status'].map({'0:ALIVE OR DEAD TUMOR FREE': 0, '1:DEAD WITH TUMOR': 1})

In [71]:
cancer_EGFR['Disease-specific Survival status'] = (
    cancer_EGFR['Disease-specific Survival status']
    .replace({
        '0:ALIVE OR DEAD TUMOR FREE': 0,
        '1:DEAD WITH TUMOR': 1
    })
)
print(cancer_EGFR['Disease-specific Survival status'].unique())

cancer_EGFR['Disease-specific Survival status'] = (
    pd.to_numeric(cancer_EGFR['Disease-specific Survival status'], errors='coerce')
)

[ 0.  1. nan]


In [72]:
#updated Disease survival (took longer than expected)
cancer_EGFR = cancer_EGFR.drop(columns=['Neoplasm Histologic Grade', 'Patient Weight', 'gender_encoded'])
cancer_EGFR.head()

,Sample ID,Patient ID,Diagnosis Age,Neoplasm Disease Stage American Joint Committee on Cancer Code,Aneuploidy Score,Buffa Hypoxia Score,Cancer Type,TCGA PanCanAtlas Cancer Type Acronym,Cancer Type Detailed,Last Communication Contact from Initial Pathologic Diagnosis Date,...,Subtype,Tumor Break Load,Tissue Prospective Collection Indicator,Tissue Retrospective Collection Indicator,Tissue Source Site,Tissue Source Site Code,TMB (nonsynonymous),Tumor Disease Anatomic Site,Tumor Type,Winter Hypoxia Score
0,TCGA-05-4402-01,TCGA-05-4402,57,STAGE IV,31,17,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,70,No,Yes,Indivumed,5,4.033333,Lung,"Lung Adenocarcinoma, Mixed Subtype",24
1,TCGA-38-6178-01,TCGA-38-6178,70,STAGE IIIA,22,-5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,448.0,...,LUAD,188,Yes,No,UNC,38,2.233333,Lung,Lung Adenocarcinoma (NOS),4
2,TCGA-44-A4SU-01,TCGA-44-A4SU,67,STAGE IA,12,-11,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,76.0,...,LUAD,90,Yes,No,Christiana Healthcare,44,4.433333,Lung,Lung Adenocarcinoma (NOS),18
3,TCGA-49-4501-01,TCGA-49-4501,67,STAGE IB,19,5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,1421.0,...,LUAD,182,No,Yes,Johns Hopkins,49,1.600000,Lung,Lung Adenocarcinoma (NOS),12
4,TCGA-50-6591-01,TCGA-50-6591,63,STAGE IV,28,21,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,167,No,Yes,University of Pittsburgh,50,1.866667,Lung,Lung Adenocarcinoma (NOS),32


In [75]:
#missing values per column
print("Missing values per column:")
print(cancer_EGFR.isnull().sum())
print(f"\nShape before removing NaN: {cancer_EGFR.shape}")
cancer_EGFR_clean = cancer_EGFR.dropna(subset=['Disease-specific Survival status', 'Diagnosis Age', 'Sex'])
print(f"Shape after removing NaN (subset approach): {cancer_EGFR_clean.shape}")

cancer_EGFR_clean.head()

Missing values per column:
Sample ID                                                                                       0
Patient ID                                                                                      0
Diagnosis Age                                                                                   0
Neoplasm Disease Stage American Joint Committee on Cancer Code                                  0
Aneuploidy Score                                                                                0
Buffa Hypoxia Score                                                                             0
Cancer Type                                                                                     0
TCGA PanCanAtlas Cancer Type Acronym                                                            0
Cancer Type Detailed                                                                            0
Last Communication Contact from Initial Pathologic Diagnosis Date                          

,Sample ID,Patient ID,Diagnosis Age,Neoplasm Disease Stage American Joint Committee on Cancer Code,Aneuploidy Score,Buffa Hypoxia Score,Cancer Type,TCGA PanCanAtlas Cancer Type Acronym,Cancer Type Detailed,Last Communication Contact from Initial Pathologic Diagnosis Date,...,Subtype,Tumor Break Load,Tissue Prospective Collection Indicator,Tissue Retrospective Collection Indicator,Tissue Source Site,Tissue Source Site Code,TMB (nonsynonymous),Tumor Disease Anatomic Site,Tumor Type,Winter Hypoxia Score
0,TCGA-05-4402-01,TCGA-05-4402,57,STAGE IV,31,17,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,70,No,Yes,Indivumed,5,4.033333,Lung,"Lung Adenocarcinoma, Mixed Subtype",24
1,TCGA-38-6178-01,TCGA-38-6178,70,STAGE IIIA,22,-5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,448.0,...,LUAD,188,Yes,No,UNC,38,2.233333,Lung,Lung Adenocarcinoma (NOS),4
2,TCGA-44-A4SU-01,TCGA-44-A4SU,67,STAGE IA,12,-11,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,76.0,...,LUAD,90,Yes,No,Christiana Healthcare,44,4.433333,Lung,Lung Adenocarcinoma (NOS),18
3,TCGA-49-4501-01,TCGA-49-4501,67,STAGE IB,19,5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,1421.0,...,LUAD,182,No,Yes,Johns Hopkins,49,1.600000,Lung,Lung Adenocarcinoma (NOS),12
4,TCGA-50-6591-01,TCGA-50-6591,63,STAGE IV,28,21,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,167,No,Yes,University of Pittsburgh,50,1.866667,Lung,Lung Adenocarcinoma (NOS),32


In [74]:
cancer_EGFR_clean.to_csv('check_EGFR.csv', index=False)
data = pd.read_csv('check_EGFR.csv')
data.head()

,Sample ID,Patient ID,Diagnosis Age,Neoplasm Disease Stage American Joint Committee on Cancer Code,Aneuploidy Score,Buffa Hypoxia Score,Cancer Type,TCGA PanCanAtlas Cancer Type Acronym,Cancer Type Detailed,Last Communication Contact from Initial Pathologic Diagnosis Date,...,Subtype,Tumor Break Load,Tissue Prospective Collection Indicator,Tissue Retrospective Collection Indicator,Tissue Source Site,Tissue Source Site Code,TMB (nonsynonymous),Tumor Disease Anatomic Site,Tumor Type,Winter Hypoxia Score
0,TCGA-05-4402-01,TCGA-05-4402,57,STAGE IV,31,17,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,70,No,Yes,Indivumed,5,4.033333,Lung,"Lung Adenocarcinoma, Mixed Subtype",24
1,TCGA-38-6178-01,TCGA-38-6178,70,STAGE IIIA,22,-5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,448.0,...,LUAD,188,Yes,No,UNC,38,2.233333,Lung,Lung Adenocarcinoma (NOS),4
2,TCGA-44-A4SU-01,TCGA-44-A4SU,67,STAGE IA,12,-11,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,76.0,...,LUAD,90,Yes,No,Christiana Healthcare,44,4.433333,Lung,Lung Adenocarcinoma (NOS),18
3,TCGA-49-4501-01,TCGA-49-4501,67,STAGE IB,19,5,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,1421.0,...,LUAD,182,No,Yes,Johns Hopkins,49,1.600000,Lung,Lung Adenocarcinoma (NOS),12
4,TCGA-50-6591-01,TCGA-50-6591,63,STAGE IV,28,21,Non-Small Cell Lung Cancer,LUAD,Lung Adenocarcinoma,NaN,...,LUAD,167,No,Yes,University of Pittsburgh,50,1.866667,Lung,Lung Adenocarcinoma (NOS),32


In [ ]:
cancer_EGFR['DSS_event'] = cancer_EGFR['Disease-specific Survival status']

print("DSS_event dtype:", cancer_EGFR['DSS_event'].dtype)
print("\nSanity check - DSS Event and Time:")
print(cancer_EGFR[['DSS_event', 'Months of disease-specific survival']].describe())

DSS_event dtype: float64

Sanity check - DSS Event and Time:
       DSS_event  Months of disease-specific survival
count  12.000000                            15.000000
mean    0.416667                            21.864966
std     0.514929                            19.367594
min     0.000000                             0.723280
25%     0.000000                             5.358845
50%     0.000000                            14.728606
75%     1.000000                            40.043397
max     1.000000                            58.454154
